In [3]:
import os
import sys
from pathlib import Path
import json
import re
import ollama
from IPython.display import Markdown, display

# ==========================================
# PASO 1: SOLUCIONAR EL ERROR DE IMPORTACIÓN
# ==========================================
# Subimos dos niveles desde 'agente_inversiones/Experimentos' para llegar a la raíz del proyecto
ruta_raiz = str(Path(os.getcwd()).resolve().parent.parent)
if ruta_raiz not in sys.path:
    sys.path.insert(0, ruta_raiz)

# ==========================================
# PASO 2: IMPORTAR TU ARQUITECTURA
# ==========================================
from agente_inversiones.agente import AgenteInversion
from agente_inversiones.modelos import EntradaInversion
from agente_inversiones.prompts_agent import SYSTEM_PROMPT, build_final_answer_prompt

agente = AgenteInversion()
modelo_llm = "qwen2.5:3b" # <-- Recuerda cambiar esto si estás usando un modelo distinto, como llama3

# ==========================================
# PASO 3: CONVERSACIÓN EN LENGUAJE NATURAL
# ==========================================
# ¡Pregúntale lo que quieras aquí!
pregunta_usuario = "¿Me conviene tomar un crédito de 50 millones al 22% efectivo anual a 60 meses?"

print("🧠 1. El LLM está analizando la pregunta y extrayendo los datos...")
prompt_extraccion = f"""
Extrae los parámetros de esta pregunta: '{pregunta_usuario}'
Devuelve ÚNICAMENTE un JSON válido con las siguientes claves y tipos de datos:
{{"monto": float, "tasa": float (en decimal), "tipo_tasa": "efectiva_anual", "periodicidad": "mensual", "plazo_periodos": int, "tipo": "credito"}}
No incluyas texto adicional ni formato markdown en tu respuesta.
"""
resp_extraccion = ollama.chat(model=modelo_llm, messages=[{"role": "user", "content": prompt_extraccion}])

# Limpiamos el JSON para evitar errores de lectura
texto_json = re.sub(r'```json\n|```', '', resp_extraccion['message']['content']).strip()

print("🧮 2. La Calculadora está ejecutando la matemática exacta (sin alucinaciones)...")
datos = json.loads(texto_json)
entrada = EntradaInversion(**datos)
resultado_matematico = agente.simular(entrada) # Aquí usamos tu fábrica y lógica exacta

print("📝 3. El Agente está redactando el informe usando tu SYSTEM_PROMPT...\n")
prompt_final = build_final_answer_prompt(
    user_question=pregunta_usuario, 
    tool_results=resultado_matematico.model_dump_json(indent=2)
)

respuesta_final = ollama.chat(
    model=modelo_llm,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT}, # ¡Usamos tus reglas estrictas!
        {"role": "user", "content": prompt_final}
    ]
)

# Mostramos el resultado renderizado
display(Markdown(respuesta_final['message']['content']))

🧠 1. El LLM está analizando la pregunta y extrayendo los datos...


2026-06-08 18:40:27,770 [INFO] httpx :: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-06-08 18:40:27,780 [INFO] agente :: ▶ INICIO :: Simulación
2026-06-08 18:40:27,788 [INFO] agente :: Registrada calculadora 'cdt' → CalculadoraCDT


🧮 2. La Calculadora está ejecutando la matemática exacta (sin alucinaciones)...


2026-06-08 18:40:28,028 [INFO] agente :: Registrada calculadora 'renta_variable' → CalculadoraRentaVariable
2026-06-08 18:40:28,032 [INFO] agente :: Registrada calculadora 'credito' → CalculadoraCredito
2026-06-08 18:40:28,034 [INFO] agente :: ▶ INICIO :: Cálculo Crédito
2026-06-08 18:40:28,036 [INFO] agente :: ✓ FIN    :: Cálculo Crédito
2026-06-08 18:40:28,037 [INFO] agente :: CalculadoraCredito.calcular ejecutado en 3.39 ms
2026-06-08 18:40:28,038 [INFO] agente :: AgenteInversion.simular ejecutado en 255.86 ms
2026-06-08 18:40:28,039 [INFO] agente :: ✓ FIN    :: Simulación


📝 3. El Agente está redactando el informe usando tu SYSTEM_PROMPT...



2026-06-08 18:41:18,974 [INFO] httpx :: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


# Resumen Ejecutivo

El análisis sugiere que un crédito de 50 millones al 22% efectivo anual a plazo fijo de 60 meses resulta en un valor futuro negativo, lo cual indica una obligación significativa para el prestamista. Los intereses totales esperados ascienden a más del 59%. Aunque la tasa nominal anual (TNA) es de 22%, la Tasa Efectiva Anual (TEA) es superior.

# Hallazgos Cuantitativos

- **Valor Futuro**: -7.96 millones
- **Rentabilidad Absoluta**: -7,956.64 millones
- **Rentabilidad Porcentual**: -59.13%
- **Tasa Efectiva Anual (TEA)**: 21.999% TEA
- **Tasa Periodica Aplicada**: 1.6710%
- **Plazo en Años**: 5 años

# Interpretación de Rentabilidad y Riesgo

## En términos de renta variable (por lo que la tasa es de efectiva anual, no aplicable directamente aquí):

### El rendimiento absoluto negativo (7.96 millones) indica una obligación financiera significativa para el usuario. Aunque la tasa nominal (22%) podría ser considerada competitiva en el mercado, la TEA superior aumenta los costos financieros.

## En términos de CDT (no aplicable aquí):

### La simulación no proporciona información relevante que sepa cómo interpretar esta tasa. El riesgo emisor está fuera del rango de análisis financiero.

# Limitaciones

- **Retención en la Fuente**: No hay datos específicos sobre la retención en la fuente del 7% sobre los rendimientos.
- **Tasas Aplicables**: La TNA proporcionada es 22%, mientras que la TEA es superior, lo cual podría afectar la efectividad de la tasa nominal.

# Proximos Analisis Sugeridos

- Sensibilidades a cambios en el tipo de crédito y plazo.
- Comparación con otros créditos o productos financieros similares.